Dataset for The Office Lines were obtained on Kaggle.

Episode Names for The Office were scraped from its wikipedia page.

IMDB scores were scraped from the IMDB website.

In [ ]:
# Loading Packages
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import csv
import re

### 1 Data Gathering

In [2]:
# Importing script
scripts = pd.read_csv("The_Office_lines.csv", header = 0)
print(scripts.shape)
scripts.head()

(59909, 7)


,id,season,episode,scene,line_text,speaker,deleted
0,1,1,1,1,All right Jim. Your quarterlies look very good...,Michael,False
1,2,1,1,1,"Oh, I told you. I couldn't close it. So...",Jim,False
2,3,1,1,1,So you've come to the master for guidance? Is ...,Michael,False
3,4,1,1,1,"Actually, you called me in here, but yeah.",Jim,False
4,5,1,1,1,"All right. Well, let me show you how it's done.",Michael,False


In [3]:
# Web Scraping The Office Episodes
# URL = "https://en.wikipedia.org/wiki/List_of_The_Office_(American_TV_series)_episodes"
# page = requests.get(URL)
# soup = BeautifulSoup(page.text, 'html.parser')

# table = soup.findAll('table',{"class":"wikiepisodetable"})[0:9]

# tr = []
# for season in table:
#     tr.append(season.findAll(['tr'])[1:])
    
# csvFile = open("office_eps.csv",'wt',newline='',encoding='utf-8')
# writer = csv.writer(csvFile)  
# try:   
#     for item in tr:
#         for cell in item:
#             th = cell.find_all('th')
#             th_data = [col.text.strip('\n') for col in th]
#             td = cell.find_all('td')
#             row = [i.text.replace('\n','') for i in td]
#             writer.writerow(th_data+row)      
        
# finally:   
#     csvFile.close()

In [4]:
episodes = pd.read_csv("office_eps.csv", header = None)

In [5]:
episodes.columns = ["Overall No", "Season Ep No", "Title", "Director", "Writer",
                    "Air Date", "Prod Code", "Viewers"]

In [6]:
episodes.head()

,Overall No,Season Ep No,Title,Director,Writer,Air Date,Prod Code,Viewers
0,1,1,"""Pilot""",Ken Kwapis,Ricky Gervais & Stephen Merchant and Greg Daniels,"March 24, 2005 (2005-03-24)",1001,11.20[16]
1,2,2,"""Diversity Day""",Ken Kwapis,B. J. Novak,"March 29, 2005 (2005-03-29)",1002,6.00[17]
2,3,3,"""Health Care""",Ken Whittingham,Paul Lieberstein,"April 5, 2005 (2005-04-05)",1006,5.80[18]
3,4,4,"""The Alliance""",Bryan Gordon,Michael Schur,"April 12, 2005 (2005-04-12)",1004,5.40[19]
4,5,5,"""Basketball""",Greg Daniels,Greg Daniels,"April 19, 2005 (2005-04-19)",1005,5.00[20]


In [7]:
# Extracting IMDB Episode Ratings

# data_set = []
# for i in range(1, 10):
#     URL = "https://www.imdb.com/title/tt0386676/episodes?season=" + str(i)
#     page = requests.get(URL)
#     soup = BeautifulSoup(page.text, 'html.parser')
    
#     eps = soup.findAll("div", {"class": "list_item"})
#     for episode in eps:
#         title = episode.findAll("a", {"itemprop": "name"})
#         rating = episode.findAll("span", {"class": "ipl-rating-star__rating"})
#         voters = episode.findAll("span", {"class": "ipl-rating-star__total-votes"})
#         row_data = [title[0].text,rating[0].text,voters[0].text[1:-1].replace(',','')]
#         row_data = [r.replace('\n','').strip() for r in row_data]
#         data_set.append(row_data)

In [8]:
# imdb = pd.DataFrame(data_set,columns=['Title','Rating','Voters'])
# imdb.head()

In [9]:
# imdb.to_csv("imdb.csv", index = False)

In [10]:
imdb = pd.read_csv("imdb.csv", header = 0)

### 2 Data Cleaning

#### Episodes DF

In [ ]:
# Helper Function
def first_number(s):
    """First episode number of a double episode, e.g. '15' -> 15, '104105' -> 104, '4546' -> 45."""
    for k in range(1, len(s)):
        if int(s[k:]) == int(s[:k]) + 1:
            return int(s[:k])
    return int(s)


In [ ]:
# Address issue of double episodes
# Double episodes have two production codes (8 digits instead of 4)
episodes = episodes.astype(str)
episodes["Is Double"] = episodes["Prod Code"].str.len() > 4
for col in ["Overall No", "Season Ep No"]:
    episodes[col] = [first_number(x) if dbl else int(x)
                     for x, dbl in zip(episodes[col], episodes["Is Double"])]

episodes["Season"] = episodes["Prod Code"].str[0].astype(int)

In [ ]:
# Clean viewers column
episodes["Viewers"] = episodes["Viewers"].str.extract(r"^(\d+\.?\d*)")[0].astype(float)

In [14]:
# Clean Air Date Column
episodes["Air Date"] = pd.to_datetime(episodes["Air Date"].str.extract(r"\((\d{4}-\d{2}-\d{2})\)")[0])

In [15]:
# Clean Title Column
episodes["Title"] = episodes["Title"].str.replace(r'["‡*†]', "", regex=True).str.strip()
episodes = episodes.drop(columns="Prod Code")

In [16]:
# Remaining title mismatches between Wikipedia and IMDb
title_fixes = {"Email Surveillance": "E-Mail Surveillance",
               "The Manager and the Salesman": "Manager and Salesman"}
episodes["IMDB Title"] = episodes["Title"].replace(title_fixes)

In [17]:
episodes.head()

,Overall No,Season Ep No,Title,Director,Writer,Air Date,Viewers,Is Double,Season,IMDB Title
0,1,1,Pilot,Ken Kwapis,Ricky Gervais & Stephen Merchant and Greg Daniels,2005-03-24,11.2,False,1,Pilot
1,2,2,Diversity Day,Ken Kwapis,B. J. Novak,2005-03-29,6.0,False,1,Diversity Day
2,3,3,Health Care,Ken Whittingham,Paul Lieberstein,2005-04-05,5.8,False,1,Health Care
3,4,4,The Alliance,Bryan Gordon,Michael Schur,2005-04-12,5.4,False,1,The Alliance
4,5,5,Basketball,Greg Daniels,Greg Daniels,2005-04-19,5.0,False,1,Basketball


#### Scripts DF

In [18]:
scripts = scripts[~scripts["deleted"]].drop(columns="deleted")

scripts["speaker"] = scripts["speaker"].str.strip()
# Map each case-variant to its most frequent spelling
most_common_spelling = (scripts.groupby(scripts["speaker"].str.lower())["speaker"]
                        .agg(lambda s: s.value_counts().idxmax()))
scripts["speaker"] = scripts["speaker"].str.lower().map(most_common_spelling)

scripts.to_csv("script_clean.csv", index=False)

#### IMDB df

In [19]:
stripped = imdb["Title"].str.replace(r":\s*Part \d+$", "", regex=True)
imdb["Title"] = stripped.where(stripped.isin(episodes.loc[episodes["Is Double"], "Title"]), imdb["Title"])
imdb = imdb.groupby("Title", as_index=False).agg(Rating=("Rating", "mean"), Voters=("Voters", "mean"))

In [20]:
imdb.head()

,Title,Rating,Voters
0,A Benihana Christmas,8.8,4235.0
1,A.A.R.M.,9.5,5221.0
2,After Hours,8.1,2700.0
3,Andy's Ancestry,7.3,2552.0
4,Andy's Play,8.2,3125.0


### Data Merging

In [21]:
# Merge Episodes dataframe with imdb dataframe
new_episodes = episodes.merge(imdb, left_on="IMDB Title", right_on="Title", how="left",
                              suffixes=("", "_imdb"), validate="many_to_one")
new_episodes = new_episodes.drop(columns=["IMDB Title", "Title_imdb"])
new_episodes = new_episodes.astype({'Rating': 'float', 'Voters': 'float'})

In [22]:
assert len(new_episodes) == len(episodes)
assert new_episodes[["Rating", "Voters"]].notna().all().all()

In [23]:
new_episodes.to_csv("episodes.csv", index = False)

In [24]:
new_episodes = pd.read_csv("episodes.csv", header = 0)

In [25]:
new_episodes.head(5)

,Overall No,Season Ep No,Title,Director,Writer,Air Date,Viewers,Is Double,Season,Rating,Voters
0,1,1,Pilot,Ken Kwapis,Ricky Gervais & Stephen Merchant and Greg Daniels,2005-03-24,11.2,False,1,7.5,6184.0
1,2,2,Diversity Day,Ken Kwapis,B. J. Novak,2005-03-29,6.0,False,1,8.3,6062.0
2,3,3,Health Care,Ken Whittingham,Paul Lieberstein,2005-04-05,5.8,False,1,7.8,5078.0
3,4,4,The Alliance,Bryan Gordon,Michael Schur,2005-04-12,5.4,False,1,8.0,4941.0
4,5,5,Basketball,Greg Daniels,Greg Daniels,2005-04-19,5.0,False,1,8.4,5465.0


#### Merging Episodes with Script

In [26]:
new_episodes["Script Ep No"] = new_episodes.groupby("Season").cumcount() + 1

lines = (scripts.groupby(["season", "episode"])
         .agg(total_lines=("id", "size"),
              michael_lines=("speaker", lambda s: (s == "Michael").sum()))
         .reset_index())

df = new_episodes.merge(lines, left_on=["Season", "Script Ep No"],
                    right_on=["season", "episode"], how="left", validate="one_to_one")
df = df.drop(columns=["season", "episode"])
assert df["total_lines"].notna().all()

df["michael_share"] = df["michael_lines"] / df["total_lines"]
df["is_superbowl"] = df["Title"] == "Stress Relief"      # aired after Super Bowl XLIII

df.to_csv("episodes_merged.csv", index=False)
print(df.shape)
df.head()

(186, 16)


,Overall No,Season Ep No,Title,Director,Writer,Air Date,Viewers,Is Double,Season,Rating,Voters,Script Ep No,total_lines,michael_lines,michael_share,is_superbowl
0,1,1,Pilot,Ken Kwapis,Ricky Gervais & Stephen Merchant and Greg Daniels,2005-03-24,11.2,False,1,7.5,6184.0,1,229,82,0.358079,False
1,2,2,Diversity Day,Ken Kwapis,B. J. Novak,2005-03-29,6.0,False,1,8.3,6062.0,2,217,81,0.373272,False
2,3,3,Health Care,Ken Whittingham,Paul Lieberstein,2005-04-05,5.8,False,1,7.8,5078.0,3,244,56,0.229508,False
3,4,4,The Alliance,Bryan Gordon,Michael Schur,2005-04-12,5.4,False,1,8.0,4941.0,4,265,79,0.298113,False
4,5,5,Basketball,Greg Daniels,Greg Daniels,2005-04-19,5.0,False,1,8.4,5465.0,5,235,106,0.451064,False


In [ ]:
# Merge Checks
ep_counts = new_episodes.groupby("Season").size()
sc_counts = scripts.groupby("season")["episode"].nunique()
assert ep_counts.equals(sc_counts.rename_axis("Season")), (ep_counts - sc_counts).dropna()

# Script episode numbers within each season are contiguous 1..N (so cumcount lines up)
assert (scripts.groupby("season")["episode"].agg(["min", "max", "nunique"])
        .eval("min == 1 and max == nunique").all())

# Episodes are in order (Script Ep No relies on this)
assert new_episodes["Overall No"].is_monotonic_increasing
assert new_episodes.groupby("Season")["Season Ep No"].apply(lambda s: s.is_monotonic_increasing).all()

# After the merge: nothing dropped on either side
assert df["total_lines"].notna().all()
assert df["total_lines"].sum() == len(scripts)       # every script line landed on exactly one episode
assert len(df) == len(new_episodes) == len(lines)    # no unmatched script episodes

# Sanity on content
assert (df["total_lines"] >= 50).all()
assert df["michael_lines"].between(0, df["total_lines"]).all()
assert (df.loc[df["Is Double"], "total_lines"] > df["total_lines"].median()).all()
